In [1]:
import sys
from pathlib import Path
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [2]:
from config import (
    BEST_HUMAN_CUES_PATH,
    SEED_EXAMPLES_PATH,
    BEST_PROMPT_BY_CATEGORY_PATH,
    FINAL_GENERATED_CUES_BY_CATEGORY_PATH,
)
from prompt_utils import sample_demo_examples, format_demo_block
from llm_utils import generate_therapy_cue
from scoring_utils import compute_proxy_score

Using local model path: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\models\Qwen2.5-1.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


In [5]:
best_human_df = pd.read_csv(BEST_HUMAN_CUES_PATH)
seed_df = pd.read_csv(SEED_EXAMPLES_PATH)

print("BEST_PROMPT_BY_CATEGORY_PATH:", BEST_PROMPT_BY_CATEGORY_PATH)
print("File exists:", BEST_PROMPT_BY_CATEGORY_PATH.exists())

best_prompt_by_category_df = pd.read_csv(BEST_PROMPT_BY_CATEGORY_PATH)

print("BEST PROMPT DF SHAPE:", best_prompt_by_category_df.shape)
display(best_prompt_by_category_df.head(20))

best_prompt_map = dict(
    zip(best_prompt_by_category_df["subcategory"], best_prompt_by_category_df["prompt_text"])
)

print("Categories with best prompts:", len(best_prompt_map))
print(sorted(best_prompt_map.keys()))

BEST_PROMPT_BY_CATEGORY_PATH: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\final\best_prompt_by_category.csv
File exists: True
BEST PROMPT DF SHAPE: (18, 10)


,subcategory,prompt_text,mean_proxy_score,ape_round,parent_prompt_id,prompt_id,prompt_family,prompt_variant,variant_bonus,selection_score
0,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,0.689594,0,NaN,交通工具_p01,category_meta,manual_prof_override,0.03,0.719594
1,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.658961,0,NaN,休閒娛樂_p01,category_meta,v1_meta_base,0.03,0.688961
2,動作,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,0.591880,0,NaN,動作_p02,category_meta,v2_discriminative,0.05,0.641880
3,動物,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動物詞彙，並生成...,1.000000,0,NaN,動物_p02,category_meta,v2_discriminative,0.05,1.050000
4,家具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.815609,0,NaN,家具_p02,category_meta,v2_discriminative,0.05,0.865609
5,廚房用品,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.828295,0,NaN,廚房用品_p01,category_meta,v1_meta_base,0.03,0.858295
6,文具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.739974,0,NaN,文具_p02,category_meta,v2_discriminative,0.05,0.789974
7,服飾&配件,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.609606,0,NaN,服飾&配件_p01,category_meta,v1_meta_base,0.03,0.639606
8,水果,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標水果詞彙，並根據...,0.410725,0,NaN,水果_p03,category_meta,v3_life_context,0.04,0.450725
9,生活用品,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標生活用品詞彙，並...,0.616680,0,NaN,生活用品_p01,category_meta,v1_meta_base,0.03,0.646680


Categories with best prompts: 18
['交通工具', '休閒娛樂', '動作', '動物', '家具', '廚房用品', '文具', '服飾&配件', '水果', '生活用品', '調味', '購物場所', '輔具', '運動場所', '金融機構', '電器用品', '食物', '餐具']


In [6]:
final_rows = []

for _, ex in tqdm(best_human_df.iterrows(), total=len(best_human_df), desc="Final generation by category"):
    subcategory = ex["subcategory"]

    if subcategory not in best_prompt_map:
        continue

    best_prompt_text = best_prompt_map[subcategory]

    demo_examples = sample_demo_examples(
        seed_df=seed_df,
        subcategory=subcategory,
        n_examples=3,
        random_state=42,
    )
    demo_block = format_demo_block(demo_examples)

    final_generated_cue = generate_therapy_cue(
        instruction=best_prompt_text,
        target_word=ex["target_word"],
        pos=ex["pos"],
        subcategory=subcategory,
        demo_block=demo_block,
    )

    score_dict = compute_proxy_score(
        generated_cue=final_generated_cue,
        target_word=ex["target_word"],
        reference_cue=ex["cue_text"],
    )

    final_rows.append({
        "target_word": ex["target_word"],
        "pos": ex["pos"],
        "main_category": ex["main_category"],
        "subcategory": subcategory,
        "reference_best_human_cue": ex["cue_text"],
        "best_prompt": best_prompt_text,
        "final_generated_cue": final_generated_cue,
        **score_dict,
    })

final_cues_df = pd.DataFrame(final_rows)

print("FINAL CUES SHAPE:", final_cues_df.shape)
display(final_cues_df.head(20))

Final generation by category: 100%|██████████| 130/130 [03:05<00:00,  1.42s/it]

FINAL CUES SHAPE: (119, 11)


,target_word,pos,main_category,subcategory,reference_best_human_cue,best_prompt,final_generated_cue,semantic_similarity,brevity_score,target_leak_penalty,proxy_score
0,上廁所,動詞,日常行為,動作,會坐/蹲在馬桶上執行動作,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,,0.000000,0.0,1.0,0.000000
1,下象棋,動詞,日常活動,休閒娛樂,需要理解不同棋子的移動規則才能下得好,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,,0.000000,0.0,1.0,0.000000
2,假牙,名詞,日常起居,輔具,剛戴上時可能會有異物感,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,需要定期檢查和磨損情況,0.193244,1.0,0.0,0.435271
3,傘,名詞,日常起居,生活用品,"用完後要晾乾,不然會發霉",你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標生活用品詞彙，並...,需要防護，防止雨水侵入,0.365362,1.0,0.0,0.555753
4,公車,名詞,日常起居,交通工具,票價便宜,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,需要刷卡上車,0.353978,1.0,0.0,0.547784
5,冰淇淋,名詞,日常飲食,食物,吃起來冰涼黏稠,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標食物詞彙，並生成...,,0.000000,0.0,1.0,0.000000
6,冰箱,名詞,日常起居,電器用品,"長方形,有兩個門",你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標電器用品詞彙，並...,冷凍食物、保持溫度,0.102371,1.0,0.0,0.371659
7,冷氣,名詞,日常起居,電器用品,開27度可以環保愛地球,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標電器用品詞彙，並...,,0.000000,0.0,1.0,0.000000
8,刷牙,動詞,日常行為,動作,正確的XX方式是由上往下、由內到外,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,,0.000000,0.0,1.0,0.000000
9,削水果,動詞,日常行為,動作,通常使用在蘋果、梨子、棗子等較硬的果皮上,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,切開水果，準備飲食時,0.649515,1.0,0.0,0.754661


In [7]:
final_cues_df.to_csv(
    FINAL_GENERATED_CUES_BY_CATEGORY_PATH,
    index=False,
    encoding="utf-8-sig"
)
print("Saved:", FINAL_GENERATED_CUES_BY_CATEGORY_PATH)

Saved: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\final\final_generated_cues_by_category.csv


In [8]:
for subcat in sorted(final_cues_df["subcategory"].unique()):
    print("=" * 100)
    print("CATEGORY:", subcat)

    best_prompt_text = final_cues_df[final_cues_df["subcategory"] == subcat]["best_prompt"].iloc[0]
    print("\nBEST PROMPT:")
    print(best_prompt_text)

    print("\nTOP GENERATED CUES:")
    display(
        final_cues_df[final_cues_df["subcategory"] == subcat][
            ["target_word", "reference_best_human_cue", "final_generated_cue", "proxy_score"]
        ].sort_values("proxy_score", ascending=False).head(10)
    )

CATEGORY: 交通工具

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並生成具體、生活化、容易聯想到答案的提示句。

我的最終目標是幫助高齡失語症患者，透過搭乘經驗與用途線索，成功說出目標詞彙。

原則：
1. 優先描述用途、搭乘方式、常見情境。
2. 避免只描述太廣泛的移動概念。
3. 優先使用高齡者熟悉的搭車經驗與日常說法。
4. 避免太書面、太技術化的描述。

提示特徵資料集（交通工具專用）：
●用途：載人、代步、通勤、旅行、上學
●情境：在路上、停靠站牌、上下班、城市裡、出遠門
●辨識特徵：固定路線、很多人一起搭、司機開、有座位、有輪子
●熟悉場景：等車、刷卡、買票、坐著移動

生成限制：
●提示句中不得出現目標詞彙。
●每句長度限於 20 個中文字以內。
●需具體、自然、容易理解。
●盡量指向單一交通工具。

輸出格式：
●每次僅生成一個提示句。
●只輸出提示句本身，不要解釋。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
53,火車,"需要購票搭乘,有分自由座和對號座",需要提前購買車票,0.859989
77,自行車,運動或代步皆適合,騎行輕鬆，享受沿途風景,0.667316
44,汽車,需要加油才能行駛,需要錢才能開動,0.651159
4,公車,票價便宜,需要刷卡上車,0.547784
33,摩托車,需要加汽油或電力才能運行,需要牌照才能上路,0.528981
115,高鐵,"透過電力驅動,行駛於專用軌道上",需要提前預約座位,0.401098


CATEGORY: 休閒娛樂

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一個簡短、具體、生活化且容易聯想到答案的提示句，不可直接說出目標詞。每次只輸出一個提示句，不要解釋。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
1,下象棋,需要理解不同棋子的移動規則才能下得好,,0.0
20,唱歌,"有時會搭配樂器,如吉他、鋼琴或卡拉OK伴奏",,0.0
28,打拳,XX能提升身體平衡感,,0.0
39,散步,"天氣涼爽時會更舒服,避免烈日或下雨天",,0.0
45,泡茶,選擇合適的茶具與水質,,0.0
63,登山,"有可能會碰到熊,要有防熊噴霧",,0.0
64,看書,會用手翻頁或摺角作為書籤,,0.0
65,看電視,用遙控器來切換頻道或調整音量,,0.0
96,購物,節日會推出促銷活動吸引客人,,0.0
98,跳舞,與拍手、踩腳或轉圈的動作結合,,0.0


CATEGORY: 動作

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成具體、容易理解、方便聯想到答案的提示句。

目標是讓高齡失語症患者透過動作情境、目的或流程聯想到該動作名稱。

原則：
1. 提示句應描述做這個動作時的典型情境。
2. 可描述目的、使用工具、動作前後會發生的事情。
3. 避免直接命令患者去做該動作。
4. 避免只說太抽象的動作概念。

提示特徵資料集（動作專用）：
●情境：早上起床後、吃飯前、洗澡後、出門前
●目的：讓身體乾淨、整理自己、完成日常生活
●工具：牙刷、剪刀、指甲剪、馬桶、毛巾
●流程：先做什麼，再做什麼
●結果：變乾淨、變整齊、完成整理

生成限制：
●提示句中不得出現目標詞彙。
●每句長度限於 20 個中文字以內。
●需具體、清楚、容易理解。
●應盡量指向單一動作。

輸出格式：
●每次僅生成一個提示句。
●只輸出提示句本身，不要解釋。

補充要求：請優先選擇最有辨識度、最能區分其他同類詞彙的特徵，避免過於廣泛的描述。

TOP GENERATED CUES:


...

,target_word,reference_best_human_cue,final_generated_cue,proxy_score
105,除濕機,會吹出熱熱的風,冷風循環，保持空調冷,0.797478
62,瓦斯爐,鍋子、鍋鏟,烤箱、爐火、煮飯,0.778112
55,烤箱,預熱、定時、溫度調整,讓食物快速回溫,0.741331
110,電話,有回撥功能,手柄、聽筒、通話,0.682779
112,電風扇,插電後即可使用,讓房子更涼快，清潔又省電,0.645629
32,插座,通電,插電、放在桌上、廚房常見,0.635623
57,熱水瓶,煮沸後會跳起來,手柄、溫暖、洗澡后,0.602855
108,電燈,會連接開關,燈光柔和，適合休息時使用,0.560483
109,電視,阿嬤打開XX,讓視頻在家中隨時隨地播放,0.462777
18,吹風機,"部分有折疊設計,方便攜帶",夏天使用、廚房常見、放桌上、冷風凍食,0.462532


CATEGORY: 食物

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標食物詞彙，並生成具體、容易理解、貼近生活的提示句，幫助患者成功說出答案。

目標是讓高齡失語症患者透過熟悉的飲食經驗聯想到該食物名稱。

原則：
1. 提示句需具體、生活化、貼近日常飲食經驗。
2. 避免抽象或過於廣泛的描述。
3. 每句應盡量指向該食物的專屬特徵。

提示特徵資料集（食物專用）：
●味道與口感：甜、鹹、酥、軟、冰、熱
●外觀特徵：片狀、圓形、白色、金黃色
●用餐情境：早餐、點心、肚子不舒服時吃
●料理與食用方式：烤、煎、塗果醬、配牛奶
●熟悉人物或生活場景：早餐店、家裡餐桌、媽媽準備

生成限制：
●提示句中不得出現目標詞彙。
●每句長度限於 20 個中文字以內。
●需具體且容易理解。
●避免太廣泛，盡量指向單一食物。

輸出格式：
●每次僅生成一個提示句。
●只輸出提示句本身，不要解釋。

補充要求：請優先選擇最有辨識度、最能區分其他同類詞彙的特徵，避免過於廣泛的描述。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
118,麵,由小麥製成的主食,一種由小麥製成的主食,0.991003
51,湯圓,一種由糯米粉製成的食品,一種甜而不咸的圓形食品,0.838449
43,水餃,"麵皮包著內餡,冷凍儲存",一種小巧玲瓏的食品,0.788164
72,米,煮後變軟可食用,一種由稻穀磨碎後製成的糧食,0.732961
107,雞排,吃起來香脆多汁,一種外觀呈片狀、金黃色的食品,0.690983
52,漢堡,西式餐點,一種夾生肉的快餐汉堡,0.661212
22,地瓜球,"用紙袋裝,用竹籤戳著吃",一種圓形的小球狀食品,0.659005
92,豆腐,可煎、炸、燉、炒,一種由大豆製成的凝固物,0.649650
113,飯糰,常作為早餐食用,一種由米和水製成的主食,0.565434
54,火鍋,冬天或聚會時特別受歡迎,,0.000000


CATEGORY: 餐具

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一個簡短、具體、生活化且容易聯想到答案的提示句，不可直接說出目標詞。每次只輸出一個提示句，不要解釋。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
50,湯匙,"嬰兒用的特別小,邊緣圓滑,方便餵食",使用時輕拿輕放，保持平穩,0.567202
14,叉子,兒童款的XX會比較圓潤,,0.000000
69,碗,可以是陶瓷、不鏽鋼、塑膠或木頭材質製成的,,0.000000


In [ ]:
output_txt = PROJECT_ROOT / "data" / "final" / "best_prompts_and_cues_by_category.txt"

with open(output_txt, "w", encoding="utf-8") as f:
    for subcat in sorted(final_cues_df["subcategory"].unique()):
        f.write("=" * 100 + "\n")
        f.write(f"CATEGORY: {subcat}\n\n")

        best_prompt_text = final_cues_df[final_cues_df["subcategory"] == subcat]["best_prompt"].iloc[0]
        f.write("BEST PROMPT:\n")
        f.write(str(best_prompt_text) + "\n\n")

        f.write("TOP GENERATED CUES:\n")
        top_rows = final_cues_df[final_cues_df["subcategory"] == subcat].sort_values(
            "proxy_score", ascending=False
        ).head(10)

        for _, r in top_rows.iterrows():
            f.write(f"- target_word: {r['target_word']}\n")
            f.write(f"  reference_best_human_cue: {r['reference_best_human_cue']}\n")
            f.write(f"  final_generated_cue: {r['final_generated_cue']}\n")
            f.write(f"  proxy_score: {r['proxy_score']}\n\n")

print("Saved:", output_txt)

CATEGORY: 交通工具

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並生成具體、生活化、容易聯想到答案的提示句。

我的最終目標是幫助高齡失語症患者，透過搭乘經驗與用途線索，成功說出目標詞彙。

原則：
1. 優先描述用途、搭乘方式、常見情境。
2. 避免只描述太廣泛的移動概念。
3. 優先使用高齡者熟悉的搭車經驗與日常說法。
4. 避免太書面、太技術化的描述。

提示特徵資料集（交通工具專用）：
●用途：載人、代步、通勤、旅行、上學
●情境：在路上、停靠站牌、上下班、城市裡、出遠門
●辨識特徵：固定路線、很多人一起搭、司機開、有座位、有輪子
●熟悉場景：等車、刷卡、買票、坐著移動

生成限制：
●提示句中不得出現目標詞彙。
●每句長度限於 20 個中文字以內。
●需具體、自然、容易理解。
●盡量指向單一交通工具。

輸出格式：
●每次僅生成一個提示句。
●只輸出提示句本身，不要解釋。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
53,火車,"需要購票搭乘,有分自由座和對號座",需要購票才能登車,0.881714
44,汽車,需要加油才能行駛,需要找到最近的公交站,0.468300
115,高鐵,"透過電力驅動,行駛於專用軌道上",需要提前預約座位,0.401098
4,公車,票價便宜,,0.000000
33,摩托車,需要加汽油或電力才能運行,,0.000000
77,自行車,運動或代步皆適合,,0.000000


CATEGORY: 休閒娛樂

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一個簡短、具體、生活化且容易聯想到答案的提示句，不可直接說出目標詞。每次只輸出一個提示句，不要解釋。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
63,登山,"有可能會碰到熊,要有防熊噴霧",在山頂遠眺，風景盡收眼底,0.389707
1,下象棋,需要理解不同棋子的移動規則才能下得好,,0.000000
20,唱歌,"有時會搭配樂器,如吉他、鋼琴或卡拉OK伴奏",,0.000000
28,打拳,XX能提升身體平衡感,,0.000000
39,散步,"天氣涼爽時會更舒服,避免烈日或下雨天",,0.000000
45,泡茶,選擇合適的茶具與水質,,0.000000
64,看書,會用手翻頁或摺角作為書籤,,0.000000
65,看電視,用遙控器來切換頻道或調整音量,,0.000000
96,購物,節日會推出促銷活動吸引客人,,0.000000
98,跳舞,與拍手、踩腳或轉圈的動作結合,,0.000000


CATEGORY: 動作

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成具體、容易理解、方便聯想到答案的提示句。

目標是讓高齡失語症患者透過動作情境、目的或流程聯想到該動作名稱。

原則：
1. 提示句應描述做這個動作時的典型情境。
2. 可描述目的、使用工具、動作前後會發生的事情。
3. 避免直接命令患者去做該動作。
4. 避免只說太抽象的動作概念。

提示特徵資料集（動作專用）：
●情境：早上起床後、吃飯前、洗澡後、出門前
●目的：讓身體乾淨、整理自己、完成日常生活
●工具：牙刷、剪刀、指甲剪、馬桶、毛巾
●流程：先做什麼，再做什麼
●結果：變乾淨、變整齊、完成整理

生成限制：
●提示句中不得出現目標詞彙。
●每句長度限於 20 個中文字以內。
●需具體、清楚、容易理解。
●應盡量指向單一動作。

輸出格式：
●每次僅生成一個提示句。
●只輸出提示句本身，不要解釋。

補充要求：請優先選擇最有辨識度、最能區分其他同類詞彙的特徵，避免過於廣泛的描述。

TOP GENERATED CUES:


...

,target_word,reference_best_human_cue,final_generated_cue,proxy_score
105,除濕機,會吹出熱熱的風,冷風循環，保持空氣清新,0.813045
111,電鍋,內鍋、外鍋、加熱板、保溫開關,煮飯、加熱快、放餐桌上,0.813035
62,瓦斯爐,鍋子、鍋鏟,爐火熊熊烤肉香,0.718258
110,電話,有回撥功能,插座、聽筒、連接線,0.665092
112,電風扇,插電後即可使用,旋轉風扇，清潔空調,0.652793
32,插座,通電,插電、放在桌上、廚房常見,0.635623
108,電燈,會連接開關,開晚上燈，照亮回家路,0.623248
109,電視,阿嬤打開XX,收看節目、娛樂消遣,0.572666
55,烤箱,預熱、定時、溫度調整,爐子、烤麵包、烘培,0.540354
47,洗衣機,會外接一根水管,手動換衣板,0.525641


CATEGORY: 食物

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標食物詞彙，並生成具體、容易理解、貼近生活的提示句，幫助患者成功說出答案。

目標是讓高齡失語症患者透過熟悉的飲食經驗聯想到該食物名稱。

原則：
1. 提示句需具體、生活化、貼近日常飲食經驗。
2. 避免抽象或過於廣泛的描述。
3. 每句應盡量指向該食物的專屬特徵。

提示特徵資料集（食物專用）：
●味道與口感：甜、鹹、酥、軟、冰、熱
●外觀特徵：片狀、圓形、白色、金黃色
●用餐情境：早餐、點心、肚子不舒服時吃
●料理與食用方式：烤、煎、塗果醬、配牛奶
●熟悉人物或生活場景：早餐店、家裡餐桌、媽媽準備

生成限制：
●提示句中不得出現目標詞彙。
●每句長度限於 20 個中文字以內。
●需具體且容易理解。
●避免太廣泛，盡量指向單一食物。

輸出格式：
●每次僅生成一個提示句。
●只輸出提示句本身，不要解釋。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
118,麵,由小麥製成的主食,一種由小麥製成的主食,0.991003
51,湯圓,一種由糯米粉製成的食品,一種圓形的小米食品,0.859512
93,豆花,一種用黃豆製成的甜點,一種由大豆製成的冷飲小吃,0.853154
43,水餃,"麵皮包著內餡,冷凍儲存",一種扁平的小米食品,0.789745
107,雞排,吃起來香脆多汁,一種外觀呈片狀、金黃色的食品,0.690983
92,豆腐,可煎、炸、燉、炒,一種白色塊狀食品,0.689811
16,吐司,拉肚子的時候會吃的,一種薄脆的早餐面包,0.681471
22,地瓜球,"用紙袋裝,用竹籤戳著吃",一種圓形、金黃色的小食品,0.638354
113,飯糰,常作為早餐食用,一種由米製作的主食,0.567622
54,火鍋,冬天或聚會時特別受歡迎,,0.000000


CATEGORY: 餐具

BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一個簡短、具體、生活化且容易聯想到答案的提示句，不可直接說出目標詞。每次只輸出一個提示句，不要解釋。

TOP GENERATED CUES:


,target_word,reference_best_human_cue,final_generated_cue,proxy_score
50,湯匙,"嬰兒用的特別小,邊緣圓滑,方便餵食",使用時輕拿輕放，保持平穩,0.567202
14,叉子,兒童款的XX會比較圓潤,,0.000000
69,碗,可以是陶瓷、不鏽鋼、塑膠或木頭材質製成的,,0.000000
